# Rugby Rulebook Cross-Reference ("Interdependency") Network Analysis

This notebook analyzes the directed, weighted cross-reference graphs built
by the interdependency-extraction notebook (`laws/` = references within the
law text only; `laws_plan_defs/` = also including "Plan"/definitions
references). For each year it computes network-level statistics — edge
count, total/average reference weight, in/out-strength inequality (Gini
coefficient), reciprocity (compared to a randomized null model), clustering,
modularity/community structure — and fits degree-distribution models
(Poisson, power law, power law with exponential cutoff) to the in/out
strength distributions.

## Requirements

```bash
pip install networkx numpy scipy matplotlib
```

## Data

This notebook expects a `data/` folder (next to the notebook) laid out as:

```
data/
├── rugby_test_matches_per_year_backfilled.csv
├── rugby_laws_per_year.csv
├── hierarchical_trees/
├── interdependencies/
│   ├── laws/               (generated by the interdependency-extraction notebook)
│   └── laws_plan_defs/     (generated by the interdependency-extraction notebook)
└── text_files/
```

Update `DATA_DIR` in the **Configuration** cell if your data lives
elsewhere.



## Imports

In [ ]:
import networkx as nx
from pathlib import Path 
import matplotlib.pyplot as plt
import matplotlib.ticker as tkr
from networkx.drawing.nx_agraph import graphviz_layout
from networkx.algorithms.community import greedy_modularity_communities
import re
import string
import numpy as np
from collections import Counter
import scipy.optimize as opt
import math
from scipy.special import factorial
from scipy.special import gammaln
from matplotlib.lines import Line2D



## Configuration

In [ ]:
DATA_DIR = Path('data')

INTERDEP_BASE = DATA_DIR / 'interdependencies'
LAWS_DIR = INTERDEP_BASE / 'laws'
LAWS_PLAN_DEFS_DIR = INTERDEP_BASE / 'laws_plan_defs'

## Load the saved cross-reference graphs

Loads every `.gexf` file in `laws_plan_defs/` into `inter_networks`
(`year -> networkx.DiGraph`).

In [ ]:
folder_path = LAWS_PLAN_DEFS_DIR

inter_networks = {}

for file_path in (folder_path.glob('*.gexf')):
    with open(file_path, 'r') as file: 
        file_name = file_path.stem
        year = file_name.replace('ints', '')

        G = nx.read_gexf(file)
        inter_networks[year] = G

Same, for the `laws/`-only graphs, into `og_inter_networks`.

In [ ]:
folder_path = LAWS_DIR

og_inter_networks = {}

for file_path in (folder_path.glob('*.gexf')):
    with open(file_path, 'r') as file: 
        file_name = file_path.stem
        year = file_name.replace('ints', '')

        G = nx.read_gexf(file)
        og_inter_networks[year] = G

## Basic edge/weight statistics

For the `laws_plan_defs` graphs: edge count, total reference weight, average
weight per existing edge, and average weight per possible dyad (i.e.
including non-edges).

In [ ]:
num_edges = {}
total_weight = {}
avg_weight_exist = {}
avg_weight_all = {}

for year, G in inter_networks.items():
    num_edges[int(year)] = G.number_of_edges()
    total_weight[int(year)] = G.size(weight="weight")
    avg_weight_exist[int(year)] = G.size(weight="weight") / max(G.number_of_edges(),1)
    avg_weight_all[int(year)] = G.size(weight='weight') / max((G.number_of_nodes()*(G.number_of_nodes()-1)),1)


Same statistics for the `laws`-only graphs.

In [ ]:
og_num_edges = {}
og_total_weight = {}
og_avg_weight_exist = {}
og_avg_weight_all = {}

for year, G in og_inter_networks.items():
    og_num_edges[int(year)] = G.number_of_edges()
    og_total_weight[int(year)] = G.size(weight="weight")
    og_avg_weight_exist[int(year)] = G.size(weight="weight") / max(G.number_of_edges(),1)
    og_avg_weight_all[int(year)] = G.size(weight='weight') / max((G.number_of_nodes()*(G.number_of_nodes()-1)),1)

Plot edge count by year for both variants.

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(num_edges.keys(), num_edges.values(), color='black', label='Laws')
plt.scatter(og_num_edges.keys(), og_num_edges.values(), color='turquoise', alpha=0.5, label='Laws, Plan, + Defs')
plt.ylabel('# of edges', size=16)
plt.xlabel('Year', size=16)
plt.legend(loc='best')

Plot total reference weight by year for both variants.

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(total_weight.keys(), total_weight.values(), color='black', label='Laws')
plt.scatter(og_total_weight.keys(), og_total_weight.values(), color='turquoise', alpha=0.5, label='Laws, Plan, + Defs')
plt.ylabel('Total weight', size=16)
plt.xlabel('Year', size=16)
plt.legend(loc='best')

Plot average weight (over existing edges, and over all possible dyads) by year for both variants.

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(avg_weight_exist.keys(), avg_weight_exist.values(), color='black', label='Laws')
plt.scatter(og_avg_weight_exist.keys(), og_avg_weight_exist.values(), color='turquoise', alpha=0.5, label='Laws, Plan, + Defs')
plt.ylabel('Average weight (existing edges)', size=16)
plt.xlabel('Year', size=16)
plt.legend(loc='best')

plt.figure(figsize=(5,5))
plt.scatter(avg_weight_all.keys(), avg_weight_all.values(), color='black', label='Laws')
plt.scatter(og_avg_weight_all.keys(), og_avg_weight_all.values(), color='turquoise', alpha=0.5, label='Laws, Plan, + Defs')
plt.ylabel('Average weight (all dyads)', size=16)
plt.xlabel('Year', size=16)
plt.legend(loc='best')

## Strength inequality (Gini coefficient)

From here on, only the `laws`-only graphs are used, with self-loops (a law
referencing itself) removed first.

In [ ]:
og_inter_networks_selfless = {}

for year, G in og_inter_networks.items():
    self_edges = list(nx.selfloop_edges(G))
    G.remove_edges_from(self_edges)
    og_inter_networks_selfless[year] = G


Compute in-strength and out-strength arrays (weighted degree) per year.

In [ ]:
in_strengths = {}
out_strengths = {}

for year, G in og_inter_networks_selfless.items():
    in_strengths[year] = np.array([strength for node, strength in G.in_degree(weight='weight')])
    out_strengths[year] = np.array([strength for node, strength in G.out_degree(weight='weight')])

Gini coefficient of a strength distribution — 0 is perfectly equal, 1 is maximally unequal.

In [ ]:
def gini_coefficient(strengths):
    strengths = np.sort(strengths.flatten())
    
    if np.sum(strengths) == 0:
        return 0.0
    
    n = strengths.shape[0]
    index = np.arange(1, n+1)
    return (np.sum((2*index - n - 1) * strengths)) / (n*np.sum(strengths))

Compute Gini coefficients for in-strength and out-strength, per year.

In [ ]:
ginis_in = {}
ginis_out = {}

for year, in_strength in in_strengths.items():
    ginis_in[int(year)] = gini_coefficient(in_strength)
    ginis_out[int(year)] = gini_coefficient(out_strengths[year])

Plot the Gini coefficient of in-strength by year.

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(ginis_in.keys(), ginis_in.values(), color='black', label='Laws')
plt.ylabel('Gini coefficient of in-strength', size=16)
plt.xlabel('Year', size=16)


Plot the Gini coefficient of out-strength by year.

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(ginis_out.keys(), ginis_out.values(), color='black', label='Laws')
plt.ylabel('Gini coefficient of out-strength', size=16)
plt.xlabel('Year', size=16)

## Weighted reciprocity vs. a random null model

`weighted_reciprocity` computes what fraction of total edge weight is
"reciprocated" (i.e. the reverse edge also exists, counting the smaller of
the two weights). `compare_reciprocity` compares the observed network's
reciprocity to that of many random directed graphs with the same node/edge
count and the same weight distribution, returning a normalized score `rho`
(0 = no more reciprocal than random, 1 = fully reciprocal) with a
bootstrap confidence interval.

In [ ]:
def weighted_reciprocity(G):
    total_weight_sum = 0
    mutual_weight_sum = 0

    for u, v, data in G.edges(data=True):
        weight = data.get('weight', 1)
        total_weight_sum += weight
        if G.has_edge(v, u):
            mutual_weight_sum += min(weight, G[v][u].get('weight', 1))

    if total_weight_sum > 0:
        return mutual_weight_sum/total_weight_sum
    else:
        return 0

def compare_reciprocity(G, iterations, seed):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    weights = []
    for u, v, data in G.edges(data=True):
        weight = data.get('weight', 1)
        weights.append(weight)
    observed_reciprocity = weighted_reciprocity(G)

    all_reciprocities = []

    for i in np.arange(iterations):
        graph_seed = int(seed + i)
        for u, v, data in G.edges(data=True):
            weight = data.get('weight', 1)
            weights.append(weight)

        R = nx.gnm_random_graph(n, m, directed=True, seed=graph_seed)
        for u, v in R.edges():
            R[u][v][weight] = np.random.choice(weights)

        all_reciprocities.append(weighted_reciprocity(R))

    random_average = np.nanmean(all_reciprocities)
    reciprocities_rho = [np.nan if r==1 else (observed_reciprocity - r)/(1-r) for r in all_reciprocities]

    rho = (observed_reciprocity-random_average)/(1-random_average)
    rho_ci = tuple(np.nanpercentile(reciprocities_rho, [2.5, 97.5]))

    return rho, rho_ci




Run the comparison (1000 random-graph draws) for every year.

In [ ]:
weighted_recip_comparison = {}

for year, G in og_inter_networks_selfless.items():
    weighted_recip_comparison[int(year)] = compare_reciprocity(G, 1000, 1) 


In [ ]:
print(weighted_recip_comparison)

Plot rho with its confidence interval by year.

In [ ]:
means = [value[0] for value in weighted_recip_comparison.values()]
ci = [value[1] for value in weighted_recip_comparison.values()]
lower = [l for l,_ in ci]
upper = [u for _,u in ci]
lower_errors = np.abs(np.asarray([mean-lower[i] for i, mean in enumerate(means)]))
upper_errors = np.abs(np.asarray([upper[i]-mean for i, mean in enumerate(means)]))
errors = np.array([lower_errors,upper_errors])

plt.figure(figsize=(5,5))
plt.errorbar(weighted_recip_comparison.keys(), means, yerr=errors, fmt='-', capsize=5, ecolor='red', linestyle='none')
plt.scatter(weighted_recip_comparison.keys(), means, color='black', label='Laws')
plt.ylabel(fr'$\rho_{{n,m}}$', size=16)
plt.xlabel('Year', size=16)


## Clustering

Global clustering (transitivity, treated as undirected), and average local
clustering both unweighted and weighted, per year.

In [ ]:
global_cc = {}
local_cc_unw = {}
local_cc_w = {}

for year, G in og_inter_networks_selfless.items():
    global_cc[int(year)] = nx.transitivity(G)
    local_cc_unw[int(year)] = nx.average_clustering(G, weight='None')
    local_cc_w[int(year)] = nx.average_clustering(G, weight='weight')

global_cc = dict(sorted(global_cc.items()))
local_cc_unw = dict(sorted(local_cc_unw.items()))
local_cc_w = dict(sorted(local_cc_w.items()))

Plot all three clustering measures by year.

In [ ]:
plt.figure(figsize=(5,5))
plt.plot(global_cc.keys(), global_cc.values(), color='green', marker='o', linestyle='--', label='Global CC (transitivity, undirected)')
plt.plot(local_cc_unw.keys(), local_cc_unw.values(), color='blue', marker='o', label='Average local CC (directed, unweighted)')
plt.plot(local_cc_w.keys(), local_cc_w.values(), color='red', marker='o', label='Average local CC (directed, weighted)')
plt.ylabel(fr'Clustering', size=16)
plt.xlabel('Year', size=16)

## Community structure

Greedy modularity-maximizing community detection, weighted by reference
count.

In [ ]:
modularity = {}
num_communities = {}

for year, G in og_inter_networks_selfless.items():
    community = greedy_modularity_communities(G, weight='weight')
    modularity[int(year)] = nx.community.modularity(G, community, weight="weight")
    num_communities[int(year)] = len(community)


Plot modularity and number of communities by year.

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(modularity.keys(), modularity.values(), color='black', label='Laws')
plt.ylabel(fr'Modularity', size=16)
plt.xlabel('Year', size=16)

plt.figure(figsize=(5,5))
plt.scatter(num_communities.keys(), num_communities.values(), color='black', label='Laws')
plt.ylabel(fr'# of communities', size=16)
plt.xlabel('Year', size=16)

## Degree-distribution model fitting

Compute in/out-strength distributions (excluding zero-weight nodes), then
fit three candidate models — Poisson, power law, and power law with an
exponential cutoff — to each year's distribution.

In [ ]:
in_degs = {}
in_degs_count = {}
out_degs = {}
out_degs_count = {}

for year, G in og_inter_networks_selfless.items():
    in_degs[year] = [d for _, d in G.in_degree(weight='weight') if d!=0]
    out_degs[year] = [d for _, d in G.out_degree(weight='weight') if d!=0]

    in_degree_dist = Counter(in_degs[year])
    in_degree_dist.pop(0, None)
    out_degree_dist = Counter(out_degs[year])
    out_degree_dist.pop(0, None)

    in_degs_count[year] = in_degree_dist
    out_degs_count[year] = out_degree_dist

The three candidate model functions, and an AIC helper to compare their fits.

In [ ]:
def poisson_fit(x, l):
    log_y = x * np.log(l) - l - gammaln(x + 1)
    #y = l**(x) * np.exp(-l) / factorial(x)
    return np.exp(log_y)

def power_law_fit(x, A, alpha):
    return A * x**(-alpha)

def power_exp_fit(x, A, l, alpha):
    y = A * np.exp(-l * x) * x**(-alpha)
    return y

In [ ]:
def aic(y_true, y_pred, k):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    n = len(y_true)
    rss = np.sum((y_true - y_pred) ** 2)
    rss = max(rss, np.finfo(float).eps)  # avoid log(0)
    return n * np.log(rss / n) + 2 * k

For each year, fits all three models (as data allows — the power law
needs at least 2 distinct values, the power-exp cutoff needs at least 3) and
picks the best one by AIC.

In [ ]:
def best_fits(degs_count):
    p_bestfits, p_covs = {}, {}
    pl_bestfits, pl_covs = {}, {}
    pel_bestfits, pel_covs = {}, {}
    best_model = {}

    for year, in_count in degs_count.items():
        count = np.array(list(in_count.keys()), dtype=float)
        freq = np.array(list(in_count.values()), dtype=float)

        order = np.argsort(count)
        count, freq = count[order], freq[order]
        prob = freq / freq.sum()

        p_aic = pl_aic = pel_aic = None

        if len(count) >= 1:
            popt, pcov = opt.curve_fit(poisson_fit, count, prob, p0=[1], maxfev=1000000)
            p_bestfits[year], p_covs[year] = popt, pcov
            p_aic = aic(prob, poisson_fit(count, *popt), len(popt))
        else:
            p_bestfits[year] = None

        if len(count) >= 2:
            popt, pcov = opt.curve_fit(power_law_fit, count, prob, p0=[1, 1], maxfev=1000000)
            pl_bestfits[year], pl_covs[year] = popt, pcov
            pl_aic = aic(prob, power_law_fit(count, *popt), len(popt))
        else:
            pl_bestfits[year] = None

        if len(count) >= 3:
            popt, pcov = opt.curve_fit(power_exp_fit, count, prob, p0=[1, 1, 1], maxfev=1000000)
            pel_bestfits[year], pel_covs[year] = popt, pcov
            pel_aic = aic(prob, power_exp_fit(count, *popt), len(popt))
        else:
            pel_bestfits[year] = None

        aics = {'Poisson': p_aic, 'Power law': pl_aic, 'PL + exp': pel_aic}
        valid = {k: v for k, v in aics.items() if v is not None}

        best_model[year] = min(valid, key=valid.get) if valid else None

    return p_bestfits, p_covs, pl_bestfits, pl_covs, pel_bestfits, pel_covs, best_model

Run the fitting for in-strength and out-strength distributions.

In [ ]:
p_bestfits_in, p_covs_in, pl_bestfits_in, pl_covs_in, pel_bestfits_in, pel_covs_in, best_model_in = best_fits(in_degs_count)
p_bestfits_out, p_covs_out, pl_bestfits_out, pl_covs_out, pel_bestfits_out, pel_covs_out, best_model_out = best_fits(out_degs_count)

Plot each year's in/out-strength histogram with all three fitted curves overlaid.

In [ ]:
def plot_hist_fits(in_degs, out_degs, p_bestfits_in, pl_bestfits_in, pel_bestfits_in, p_bestfits_out, pl_bestfits_out, pel_bestfits_out):
    for year, dist in in_degs.items():
        fig, ax = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)
        fig.suptitle(year)
        

        for i, (data, fits, label) in enumerate([
            (dist,            (p_bestfits_in, pl_bestfits_in, pel_bestfits_in), 'In-weight'),
            (out_degs[year],  (p_bestfits_out, pl_bestfits_out, pel_bestfits_out), 'Out-weight'),
        ]):
            if len(data) == 0:
                continue

            bins = np.arange(min(data) - 0.5, max(data) + 1.5, 1)
            ax[i].hist(data, bins=bins, edgecolor='black', color='white',
                    rwidth=0.8, density=True)
            ax[i].set_xlabel(label, size=10)
            ax[i].set_xlim(0)
            ax[i].xaxis.set_major_locator(tkr.MaxNLocator(integer=True))
            ax[i].set_xlabel(label, size=10)

            p_fit, pl_fit, pel_fit = (f.get(year) for f in fits)

            x_int = np.arange(1, max(data) + 1)
            if p_fit is not None:
                ax[i].plot(x_int, poisson_fit(x_int, *p_fit),
                        'o-', ms=3, lw=1, label=rf'Poisson, $\lambda=${round(p_fit[0],2)}')

            x_smooth = np.linspace(1, max(data), 300)
            if pl_fit is not None:
                ax[i].plot(x_smooth, power_law_fit(x_smooth, *pl_fit),
                        '--', lw=1, label=rf'Power law, $\alpha=${round(pl_fit[1],2)}')
            if pel_fit is not None:
                ax[i].plot(x_smooth, power_exp_fit(x_smooth, *pel_fit),
                        ':', lw=1.5, label=rf'Power-exp, $\lambda=${round(pel_fit[1],2)}, $\alpha=${round(pel_fit[2],2)}')

            ax[i].legend(fontsize=7)

        ax[0].set_ylabel('Probability', size=10)
        fig.tight_layout()
        plt.show()

In [ ]:
plot_hist_fits(in_degs, out_degs, p_bestfits_in, pl_bestfits_in, pel_bestfits_in, p_bestfits_out, pl_bestfits_out, pel_bestfits_out)

## Timeline of best-fitting model

In [ ]:
def plot_best_fits_timeline(best_model_in, best_model_out):
    years = np.array(list(best_model_in.keys()), dtype=float)
    in_models = np.array(list(best_model_in.values()), dtype=str)
    order = np.argsort(years)
    years, in_models = years[order], in_models[order]

    out_models = np.array(list(best_model_out.values()), dtype=str)
    order = np.argsort(years)
    years, out_models = years[order], out_models[order]

    years_t = years.copy()
    years_t = np.append(years_t, 2027)
    difference = [years[i+1]-year for i, year in enumerate(years_t) if i+2<len(years_t)]

    fig, ax = plt.subplots(2, 1, figsize=(8, 3), sharex=True)

    for i, model in enumerate(in_models):
        if model=='Poisson':
            in_color='green'
        elif model=='Power law':
            in_color='red'
        else:
            in_color='yellow'
        ax[0].axvspan(years[i], years_t[i+1]-.5, color=in_color, alpha=0.5, zorder=0)

    for i, model in enumerate(out_models):
        if model=='Poisson':
            out_color='green'
        elif model=='Power law':
            out_color='red'
        else:
            out_color='yellow'
        ax[1].axvspan(years[i], years_t[i+1]-.5, color=out_color, alpha=0.5, zorder=0)


    ax[0].set_xlim((1874, 2027))
    ax[0].set_ylabel('In-weight', rotation=0, labelpad=30)
    ax[0].set_yticks([])

    ax[1].set_xlabel('Year')
    ax[1].set_ylabel('Out-weight', rotation=0, labelpad=30)
    ax[1].set_yticks([])

    custom_handles = [
        Line2D([], [], color='green', lw=6, label='Poisson'),
        Line2D([], [], color='red', lw=6, linestyle='--', label='Power law'),
        Line2D([], [], color='yellow', lw=6, label='PL + exp')
    ]

    labels = [h.get_label() for h in custom_handles]
    ax[1].legend(handles=custom_handles, labels=labels, loc=(0.25,-1), ncol=3)


    plt.tight_layout()

In [ ]:
plot_best_fits_timeline(best_model_in, best_model_out)